# GEPA Prompt Optimization: Legal Demand Assistant

Faithful implementation of the GEPA (Genetic-Pareto) methodology for prompt optimization.

**Core GEPA features implemented:**
- **Population-based search** — maintains multiple candidate prompts per task
- **Pareto frontier** — tracks candidates that excel on different query subsets
- **LLM-driven reflective mutation** — model reads execution traces, diagnoses failures, proposes improved prompts
- **Merge strategy** — combines complementary candidates from the frontier
- **Multi-objective scoring** — separate format, content, and completeness objectives
- **Minibatch evaluation** — evaluates on subsets, full eval only on acceptance
- **Evaluation caching** — avoids redundant (candidate, query) evaluations
- **Strict acceptance** — new candidates accepted only if they improve on subsample

Uses Qwen3.5-9B (open source, no API keys required).

**Requirements:** Colab with GPU (A100 recommended)

## Section 0: Install Dependencies

In [ ]:
!pip install transformers>=4.46.0 bitsandbytes>=0.44.0 accelerate

## Section 1: Imports and Configuration

In [ ]:
import json
import copy
import random
import hashlib
from datetime import datetime
from collections import defaultdict

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3.5-9B"
OUTPUT_FILE = "./gepa_optimized_prompts.json"

# GEPA hyperparameters
NUM_GENERATIONS = 3         # Evolution iterations
POPULATION_SIZE = 3         # Candidates maintained per task type
MINIBATCH_SIZE = 2          # Queries per minibatch for mutation eval
MERGE_PROBABILITY = 0.4     # Chance of attempting merge after a successful mutation
EPSILON_GREEDY = 0.2        # Exploration rate for candidate selection

random.seed(42)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Section 2: Load Model

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    attn_implementation="eager",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()
print(f"Model loaded: {MODEL_ID}")

## Section 3: Core Infrastructure

Generation, multi-objective scoring, evaluation caching, and Pareto frontier management.

In [ ]:
def generate(system_prompt, user_input, max_tokens=512, temperature=0.7):
    """Generate a response from the model."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_tokens, temperature=temperature,
            top_p=0.9, do_sample=temperature > 0,
        )
    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()


def candidate_id(prompt_text):
    """Generate a short hash ID for a candidate prompt."""
    return hashlib.md5(prompt_text.encode()).hexdigest()[:8]

In [ ]:
# ── Multi-Objective Scoring ──────────────────────────────────────────────────
#
# Three objectives (like GEPA's multi-objective Pareto):
#   1. Format compliance  — does it follow the expected output structure?
#   2. Content quality    — is the legal content correct and relevant?
#   3. Completeness       — are all required elements present?

OBJECTIVE_NAMES = ["format", "content", "completeness"]

# Map each criterion to an objective
CRITERION_TO_OBJECTIVE = {
    # Format
    "json_format": "format",
    "formal_tone": "format",
    # Content
    "correct_claim_type": "content",
    "legal_basis": "content",
    "reasoning_provided": "content",
    "values_correct": "content",
    "correctly_identifies_inadequate": "content",
    "correctly_identifies_adequate": "content",
    "strengths_listed": "content",
    # Completeness
    "parties_identified": "completeness",
    "amount_stated": "completeness",
    "deadline_included": "completeness",
    "escalation_mentioned": "completeness",
    "all_fields_extracted": "completeness",
    "missing_elements_listed": "completeness",
    "improvements_suggested": "completeness",
    "multiple_remedies": "completeness",
}


def score_response(response, criteria):
    """Score a response. Returns per-criterion bools and per-objective scores."""
    scores = {}
    resp_lower = response.lower()

    for c in criteria:
        if c == "formal_tone":
            scores[c] = any(w in resp_lower for w in ["dear", "sincerely", "represent", "i represent"])
        elif c == "parties_identified":
            scores[c] = "[" in response or any(w in resp_lower for w in ["my client", "claimant", "tenant", "freelancer"])
        elif c == "amount_stated":
            scores[c] = "$" in response
        elif c == "deadline_included":
            scores[c] = any(w in resp_lower for w in ["days", "deadline", "within"])
        elif c == "escalation_mentioned":
            scores[c] = any(w in resp_lower for w in ["legal action", "legal remedies", "pursue", "file", "court"])
        elif c == "legal_basis":
            scores[c] = any(w in resp_lower for w in ["breach", "violation", "obligation", "liability", "negligence", "statute"])
        elif c == "correct_claim_type":
            scores[c] = any(w in resp_lower for w in ["breach", "fraud", "misrepresentation", "negligence", "violation"])
        elif c == "json_format":
            scores[c] = "{" in response and "}" in response
        elif c == "reasoning_provided":
            scores[c] = any(w in resp_lower for w in ["reasoning", "because", "the ", "failed", "accepted"])
        elif c == "all_fields_extracted":
            scores[c] = all(w in resp_lower for w in ["claimant", "recipient", "damages", "deadline"])
        elif c == "values_correct":
            scores[c] = "$" in response
        elif c == "correctly_identifies_inadequate":
            scores[c] = any(w in resp_lower for w in ["false", "inadequate", "not adequate", "incomplete", "missing"])
        elif c == "correctly_identifies_adequate":
            scores[c] = any(w in resp_lower for w in ["true", "adequate", "effective", "complete", "sufficient"])
        elif c == "missing_elements_listed":
            scores[c] = any(w in resp_lower for w in ["missing", "lacks", "does not include", "no "])
        elif c == "improvements_suggested":
            scores[c] = any(w in resp_lower for w in ["should", "add", "include", "specify", "improve"])
        elif c == "strengths_listed":
            scores[c] = any(w in resp_lower for w in ["strength", "identifies", "includes", "states", "provides"])
        elif c == "multiple_remedies":
            scores[c] = resp_lower.count("remedy") + resp_lower.count("refund") + resp_lower.count("payment") + resp_lower.count("reimbursement") >= 2
        else:
            scores[c] = True

    # Compute per-objective scores
    obj_scores = {}
    for obj in OBJECTIVE_NAMES:
        obj_criteria = [c for c in criteria if CRITERION_TO_OBJECTIVE.get(c) == obj]
        if obj_criteria:
            obj_scores[obj] = sum(scores[c] for c in obj_criteria) / len(obj_criteria)
        else:
            obj_scores[obj] = 1.0  # No criteria for this objective = perfect

    total = sum(scores.values()) / len(scores) if scores else 0
    return scores, obj_scores, total

In [ ]:
# ── Evaluation Cache ──────────────────────────────────────────────────────────
#
# Caches (candidate_id, query_id) → result to avoid redundant evaluations.
# Mirrors GEPA's EvaluationCache.

class EvaluationCache:
    def __init__(self):
        self.cache = {}  # (cid, qid) → {response, scores, obj_scores, total}
        self.hits = 0
        self.misses = 0

    def get(self, cid, qid):
        key = (cid, qid)
        if key in self.cache:
            self.hits += 1
            return self.cache[key]
        self.misses += 1
        return None

    def put(self, cid, qid, result):
        self.cache[(cid, qid)] = result

    def stats(self):
        total = self.hits + self.misses
        rate = self.hits / total if total > 0 else 0
        return f"Cache: {self.hits}/{total} hits ({rate:.0%})"


eval_cache = EvaluationCache()


def evaluate_candidate(prompt_text, queries, use_cache=True, capture_traces=False):
    """
    Evaluate a candidate prompt on a set of queries.
    
    capture_traces: if True, include full response text for reflection.
                    Trace results are NOT cached (matches GEPA behavior).
    """
    cid = candidate_id(prompt_text)
    results = []

    for q in queries:
        cached = eval_cache.get(cid, q["id"]) if (use_cache and not capture_traces) else None

        if cached:
            results.append(cached)
        else:
            response = generate(prompt_text, q["input"], max_tokens=256)
            scores, obj_scores, total = score_response(response, q["criteria"])
            result = {
                "query_id": q["id"],
                "input": q["input"],
                "response": response if capture_traces else response[:100],
                "criteria_scores": scores,
                "objective_scores": obj_scores,
                "total_score": total,
            }
            results.append(result)
            if not capture_traces:
                # Only cache non-trace evaluations (matches GEPA)
                eval_cache.put(cid, q["id"], result)

    # Aggregate scores
    avg_total = sum(r["total_score"] for r in results) / len(results)
    avg_objectives = {}
    for obj in OBJECTIVE_NAMES:
        vals = [r["objective_scores"].get(obj, 1.0) for r in results]
        avg_objectives[obj] = sum(vals) / len(vals)

    return results, avg_total, avg_objectives

In [ ]:
# ── Pareto Frontier ───────────────────────────────────────────────────────────
#
# Tracks the instance frontier (per-query best) and objective frontier
# (per-objective best). Mirrors GEPA's hybrid frontier.

class ParetoFrontier:
    def __init__(self):
        # Instance frontier: query_id → (best_score, candidate_id, prompt_text)
        self.instance_front = {}
        # Objective frontier: objective_name → (best_score, candidate_id, prompt_text)
        self.objective_front = {}
        # All candidates: candidate_id → {prompt, per_query_scores, objective_scores, total}
        self.candidates = {}
        # Lineage: candidate_id → parent_candidate_id
        self.lineage = {}

    def add_candidate(self, prompt_text, query_results, avg_total, avg_objectives, parent_id=None):
        """Add a candidate and update frontiers."""
        cid = candidate_id(prompt_text)
        self.candidates[cid] = {
            "prompt": prompt_text,
            "per_query": {r["query_id"]: r["total_score"] for r in query_results},
            "objectives": avg_objectives,
            "total": avg_total,
        }
        if parent_id:
            self.lineage[cid] = parent_id

        # Update instance frontier
        for r in query_results:
            qid = r["query_id"]
            if qid not in self.instance_front or r["total_score"] > self.instance_front[qid][0]:
                self.instance_front[qid] = (r["total_score"], cid, prompt_text)

        # Update objective frontier
        for obj, score in avg_objectives.items():
            if obj not in self.objective_front or score > self.objective_front[obj][0]:
                self.objective_front[obj] = (score, cid, prompt_text)

        return cid

    def select_candidate(self, epsilon=EPSILON_GREEDY):
        """
        Epsilon-greedy candidate selection from Pareto frontier.
        With probability epsilon: random candidate.
        Otherwise: sample from frontier, weighted by how many queries/objectives it leads.
        """
        if not self.candidates:
            return None, None

        if random.random() < epsilon:
            # Exploration: random candidate
            cid = random.choice(list(self.candidates.keys()))
            return cid, self.candidates[cid]["prompt"]

        # Exploitation: weight by frontier dominance frequency
        frontier_counts = defaultdict(int)
        for _, (_, cid, _) in self.instance_front.items():
            frontier_counts[cid] += 1
        for _, (_, cid, _) in self.objective_front.items():
            frontier_counts[cid] += 1

        if not frontier_counts:
            cid = random.choice(list(self.candidates.keys()))
            return cid, self.candidates[cid]["prompt"]

        cids = list(frontier_counts.keys())
        weights = [frontier_counts[c] for c in cids]
        total_w = sum(weights)
        weights = [w / total_w for w in weights]
        cid = random.choices(cids, weights=weights, k=1)[0]
        return cid, self.candidates[cid]["prompt"]

    def get_best_overall(self):
        """Return the candidate with highest total score."""
        if not self.candidates:
            return None, None
        best_cid = max(self.candidates, key=lambda c: self.candidates[c]["total"])
        return best_cid, self.candidates[best_cid]["prompt"]

    def find_merge_pair(self):
        """
        Find two frontier candidates that are complementary:
        each excels on different queries or objectives.
        """
        frontier_cids = set()
        for _, (_, cid, _) in self.instance_front.items():
            frontier_cids.add(cid)
        for _, (_, cid, _) in self.objective_front.items():
            frontier_cids.add(cid)

        frontier_cids = list(frontier_cids)
        if len(frontier_cids) < 2:
            return None, None

        # Pick two candidates that lead on different queries
        best_pair = None
        best_complementarity = -1
        for i in range(len(frontier_cids)):
            for j in range(i + 1, len(frontier_cids)):
                c1, c2 = frontier_cids[i], frontier_cids[j]
                s1 = self.candidates[c1]["per_query"]
                s2 = self.candidates[c2]["per_query"]
                # Complementarity: how often each beats the other
                common_qs = set(s1.keys()) & set(s2.keys())
                if not common_qs:
                    continue
                c1_wins = sum(1 for q in common_qs if s1[q] > s2[q])
                c2_wins = sum(1 for q in common_qs if s2[q] > s1[q])
                comp = min(c1_wins, c2_wins)  # Higher = more complementary
                if comp > best_complementarity:
                    best_complementarity = comp
                    best_pair = (c1, c2)

        return best_pair if best_pair else (None, None)

    def summary(self):
        """Print frontier summary."""
        print(f"    Population: {len(self.candidates)} candidates")
        print(f"    Instance frontier: {len(set(cid for _, (_, cid, _) in self.instance_front.items()))} unique leaders")
        print(f"    Objective frontier:")
        for obj, (score, cid, _) in self.objective_front.items():
            print(f"      {obj}: {score:.0%} (candidate {cid})")

In [ ]:
# ── Minibatch Sampler ─────────────────────────────────────────────────────────
#
# Epoch-shuffled minibatch sampling (matches GEPA's EpochShuffledBatchSampler).

class MinibatchSampler:
    def __init__(self, queries, batch_size):
        self.queries = queries
        self.batch_size = min(batch_size, len(queries))
        self.order = list(range(len(queries)))
        random.shuffle(self.order)
        self.position = 0

    def sample(self):
        """Get next minibatch, reshuffling at epoch boundary."""
        if self.position + self.batch_size > len(self.order):
            random.shuffle(self.order)
            self.position = 0

        indices = self.order[self.position:self.position + self.batch_size]
        self.position += self.batch_size
        return [self.queries[i] for i in indices]

## Section 4: Reflective Mutation and Merge

The core GEPA operators:
- **Reflective mutation**: evaluate with traces → LLM diagnoses failures → LLM proposes fix
- **Merge**: combine two complementary frontier candidates

In [ ]:
def reflect(prompt_text, trace_results, task_type):
    """
    LLM-based reflection on execution traces.
    Reads full responses and failure patterns to diagnose issues.
    Mirrors GEPA's Actionable Side Information.
    """
    traces = ""
    for r in trace_results:
        failed = [c for c, passed in r["criteria_scores"].items() if not passed]
        passed = [c for c, p in r["criteria_scores"].items() if p]
        obj_detail = ", ".join(f"{k}: {v:.0%}" for k, v in r["objective_scores"].items())
        traces += (
            f"\n--- Query: {r['input'][:150]} ---\n"
            f"Response: {r['response'][:300]}\n"
            f"Passed: {passed}\n"
            f"Failed: {failed}\n"
            f"Objectives: {obj_detail}\n"
        )

    reflection_input = (
        f"TASK TYPE: {task_type}\n\n"
        f"CURRENT SYSTEM PROMPT:\n{prompt_text}\n\n"
        f"EXECUTION TRACES WITH SCORES:\n{traces}\n\n"
        "You are analyzing why this system prompt produced failures. "
        "For each failed criterion, explain the specific gap in the system prompt "
        "that caused the failure. Then list the top 3 changes needed, ordered by impact. "
        "Be concrete — say exactly what instruction is missing or wrong."
    )

    return generate(
        "You are a prompt engineering expert who diagnoses prompt failures from execution traces.",
        reflection_input,
        max_tokens=400,
        temperature=0.3,
    )


def mutate(current_prompt, reflection, task_type):
    """
    LLM-based mutation informed by reflection.
    Proposes an improved prompt that addresses diagnosed failures.
    """
    mutate_input = (
        f"TASK TYPE: {task_type}\n\n"
        f"CURRENT SYSTEM PROMPT:\n{current_prompt}\n\n"
        f"FAILURE ANALYSIS:\n{reflection}\n\n"
        "Write an IMPROVED system prompt that fixes the failures above.\n"
        "Rules:\n"
        "- Output ONLY the new system prompt, nothing else\n"
        "- Keep it under 300 words\n"
        "- Be specific about required output format\n"
        "- Add explicit constraints to prevent each identified failure\n"
        "- Do NOT include preamble like 'Here is the improved prompt'\n"
        "- Start directly with the prompt content"
    )

    new_prompt = generate(
        "You are a prompt engineer. Output only the requested prompt text.",
        mutate_input,
        max_tokens=500,
        temperature=0.7,
    )

    # Clean up wrapper artifacts
    new_prompt = new_prompt.strip('"').strip("'").strip("`").strip()
    if new_prompt.startswith("```"):
        lines = new_prompt.split("\n")
        new_prompt = "\n".join(lines[1:-1] if lines[-1].startswith("```") else lines[1:])

    return new_prompt.strip()


def merge_prompts(prompt_a, prompt_b, task_type):
    """
    Merge two complementary prompts into one.
    Mirrors GEPA's system-aware recombination.
    """
    merge_input = (
        f"TASK TYPE: {task_type}\n\n"
        f"PROMPT A:\n{prompt_a}\n\n"
        f"PROMPT B:\n{prompt_b}\n\n"
        "These two prompts each excel on different aspects of the task. "
        "Combine their strengths into a single improved prompt.\n"
        "Rules:\n"
        "- Output ONLY the merged prompt, nothing else\n"
        "- Keep the best instructions from each parent\n"
        "- Resolve contradictions by picking the more specific instruction\n"
        "- Keep it under 300 words\n"
        "- Start directly with the prompt content"
    )

    merged = generate(
        "You are a prompt engineer. Merge two prompts into one. Output only the merged prompt.",
        merge_input,
        max_tokens=500,
        temperature=0.5,
    )

    merged = merged.strip('"').strip("'").strip("`").strip()
    if merged.startswith("```"):
        lines = merged.split("\n")
        merged = "\n".join(lines[1:-1] if lines[-1].startswith("```") else lines[1:])

    return merged.strip()

## Section 5: Evaluation Queries

In [ ]:
EVAL_QUERIES = {
    "draft_demand_letter": [
        {
            "id": "Q1",
            "input": "Draft a demand letter for a tenant whose landlord withheld a $2,500 security deposit without providing an itemized statement of damages.",
            "criteria": ["formal_tone", "parties_identified", "amount_stated", "deadline_included", "escalation_mentioned", "legal_basis"]
        },
        {
            "id": "Q2",
            "input": "Draft a demand letter for a freelancer who completed branding work for a client, delivered the files, and never received the agreed $3,200 payment.",
            "criteria": ["formal_tone", "parties_identified", "amount_stated", "deadline_included", "escalation_mentioned", "legal_basis"]
        },
    ],
    "identify_claim": [
        {
            "id": "Q3",
            "input": "Identify the strongest legal claim where a contractor accepted an $8,000 payment, performed minimal work, and abandoned the job.",
            "criteria": ["correct_claim_type", "json_format", "reasoning_provided"]
        },
        {
            "id": "Q4",
            "input": "Identify the likely legal claim where a seller knowingly lied about a car having no accident history, and the buyer later discovered major prior damage.",
            "criteria": ["correct_claim_type", "json_format", "reasoning_provided"]
        },
    ],
    "extract_elements": [
        {
            "id": "Q5",
            "input": "Extract the claimant, recipient, damages, deadline, and requested remedy from the following demand letter:\n\nDear Mr. Allen: My client, Sarah Kim, seeks payment of $4,800 for water damage caused to her property on January 8, 2026, when your negligence led to a plumbing overflow from your unit. Please remit payment within 14 days of this letter to avoid further action.",
            "criteria": ["all_fields_extracted", "json_format", "values_correct"]
        },
        {
            "id": "Q6",
            "input": "Extract the key legal elements from the following demand letter:\n\nDear ABC Services, I represent Daniel Ortiz regarding unpaid wages in the amount of $1,950 for work completed during December 2025. Unless payment is received within 7 days, my client will consider further legal action.",
            "criteria": ["all_fields_extracted", "json_format", "values_correct"]
        },
    ],
    "evaluate_letter": [
        {
            "id": "Q7",
            "input": "Evaluate whether this demand letter is complete and effective:\n\nYou owe me money. Please fix this immediately.",
            "criteria": ["correctly_identifies_inadequate", "json_format", "missing_elements_listed", "improvements_suggested"]
        },
        {
            "id": "Q8",
            "input": "Evaluate whether this demand letter is effective:\n\nDear Property Manager, my client seeks return of a $1,800 security deposit for Apartment 2B. The tenant vacated the unit on February 1, 2026, and no itemized deductions were provided. Please return the deposit within 10 days or my client may pursue legal remedies.",
            "criteria": ["correctly_identifies_adequate", "json_format", "strengths_listed"]
        },
    ],
    "recommend_remedy": [
        {
            "id": "Q9",
            "input": "Suggest appropriate remedies when a contractor took a deposit, abandoned a home repair project, and the homeowner had to hire another contractor for additional cost.",
            "criteria": ["multiple_remedies", "json_format", "reasoning_provided"]
        },
        {
            "id": "Q10",
            "input": "Suggest remedies for a consumer who purchased a defective appliance, repeatedly requested repair or refund, and received no response from the seller.",
            "criteria": ["multiple_remedies", "json_format", "reasoning_provided"]
        },
    ],
}

## Section 6: Seed Population (Generation 0)

Initialize the population with seed prompts and run baseline evaluation.

In [ ]:
# Seed prompts — minimal starting points for evolution
SEED_PROMPTS = {
    "draft_demand_letter": [
        "You are a helpful legal assistant.",
        "You are a lawyer who writes demand letters.",
        "You draft formal legal correspondence on behalf of clients.",
    ],
    "identify_claim": [
        "You are a helpful legal assistant.",
        "You are a legal analyst who classifies disputes.",
        "You identify legal claims from factual scenarios and respond in JSON.",
    ],
    "extract_elements": [
        "You are a helpful legal assistant.",
        "You are a legal document analyst.",
        "You extract structured data from legal letters and respond in JSON.",
    ],
    "evaluate_letter": [
        "You are a helpful legal assistant.",
        "You evaluate whether demand letters are complete and effective.",
        "You are a legal quality reviewer who assesses demand letters and responds in JSON.",
    ],
    "recommend_remedy": [
        "You are a helpful legal assistant.",
        "You recommend legal remedies for civil disputes.",
        "You suggest remedies for legal disputes and respond in JSON format.",
    ],
}

# Initialize frontiers and evaluate seed population
frontiers = {}
samplers = {}

print("=" * 60)
print("GENERATION 0 — Seed Population Evaluation")
print("=" * 60)

baseline_scores = {}

for task_type, queries in EVAL_QUERIES.items():
    print(f"\n  [{task_type}]")
    frontier = ParetoFrontier()
    samplers[task_type] = MinibatchSampler(queries, MINIBATCH_SIZE)

    for i, seed in enumerate(SEED_PROMPTS[task_type]):
        results, avg_total, avg_obj = evaluate_candidate(seed, queries)
        cid = frontier.add_candidate(seed, results, avg_total, avg_obj)
        print(f"    Seed {i} ({cid}): {avg_total:.0%} | "
              f"fmt={avg_obj.get('format', 0):.0%} "
              f"cnt={avg_obj.get('content', 0):.0%} "
              f"cmp={avg_obj.get('completeness', 0):.0%}")

    frontier.summary()
    frontiers[task_type] = frontier

    # Track baseline as best seed score
    _, best_prompt = frontier.get_best_overall()
    baseline_scores[task_type] = frontier.candidates[frontier.get_best_overall()[0]]["total"]

baseline_overall = sum(baseline_scores.values()) / len(baseline_scores)
print(f"\nGen 0 Overall: {baseline_overall:.0%}")
print(eval_cache.stats())

## Section 7: GEPA Evolution Loop

The main loop, faithful to GEPA's engine:

```
FOR each generation:
  FOR each task type:
    (1) SELECT candidate from Pareto frontier (epsilon-greedy)
    (2) SAMPLE minibatch from queries
    (3) EVALUATE candidate on minibatch WITH traces
    (4) REFLECT — LLM reads traces, diagnoses failures
    (5) MUTATE — LLM proposes improved prompt
    (6) EVALUATE new candidate on same minibatch WITHOUT traces
    (7) ACCEPT if new_score > old_score on subsample
    (8) If accepted: FULL EVAL on all queries, update frontier
    (9) ATTEMPT MERGE if scheduled (after successful mutation)
```

In [ ]:
evolution_trace = []  # Full log of every iteration

for gen in range(1, NUM_GENERATIONS + 1):
    print(f"\n{'=' * 60}")
    print(f"GENERATION {gen}")
    print(f"{'=' * 60}")

    last_accepted = {}  # Track which tasks had an accepted mutation this gen

    for task_type, queries in EVAL_QUERIES.items():
        print(f"\n  [{task_type}]")
        frontier = frontiers[task_type]
        sampler = samplers[task_type]

        # ── (1) SELECT from Pareto frontier ──
        parent_cid, parent_prompt = frontier.select_candidate()
        print(f"    Selected: {parent_cid}")

        # ── (2) SAMPLE minibatch ──
        minibatch = sampler.sample()
        mb_ids = [q["id"] for q in minibatch]
        print(f"    Minibatch: {mb_ids}")

        # ── (3) EVALUATE on minibatch WITH traces ──
        trace_results, parent_mb_score, _ = evaluate_candidate(
            parent_prompt, minibatch, use_cache=False, capture_traces=True
        )
        print(f"    Parent minibatch score: {parent_mb_score:.0%}")

        # Skip if already perfect on this minibatch
        if parent_mb_score >= 1.0:
            print(f"    Perfect score — skipping mutation")
            evolution_trace.append({
                "gen": gen, "task": task_type, "action": "skip_perfect",
                "parent": parent_cid, "score": parent_mb_score,
            })
            continue

        # ── (4) REFLECT — LLM diagnoses failures ──
        print(f"    Reflecting...")
        reflection = reflect(parent_prompt, trace_results, task_type)
        print(f"    Reflection: {reflection[:120]}...")

        # ── (5) MUTATE — LLM proposes improved prompt ──
        print(f"    Mutating...")
        child_prompt = mutate(parent_prompt, reflection, task_type)
        child_cid = candidate_id(child_prompt)
        print(f"    Child {child_cid}: {child_prompt[:80]}...")

        # ── (6) EVALUATE child on SAME minibatch WITHOUT traces ──
        child_mb_results, child_mb_score, _ = evaluate_candidate(
            child_prompt, minibatch, use_cache=False, capture_traces=False
        )
        print(f"    Child minibatch score: {child_mb_score:.0%}")

        # ── (7) ACCEPT if strictly better on subsample ──
        if child_mb_score > parent_mb_score:
            print(f"    ✓ Accepted: {parent_mb_score:.0%} → {child_mb_score:.0%}")

            # ── (8) FULL EVAL on all queries, update frontier ──
            print(f"    Running full evaluation...")
            full_results, full_total, full_obj = evaluate_candidate(child_prompt, queries)
            frontier.add_candidate(child_prompt, full_results, full_total, full_obj, parent_id=parent_cid)
            print(f"    Full score: {full_total:.0%} | "
                  f"fmt={full_obj.get('format', 0):.0%} "
                  f"cnt={full_obj.get('content', 0):.0%} "
                  f"cmp={full_obj.get('completeness', 0):.0%}")
            last_accepted[task_type] = True

            evolution_trace.append({
                "gen": gen, "task": task_type, "action": "accepted",
                "parent": parent_cid, "child": child_cid,
                "parent_mb_score": parent_mb_score, "child_mb_score": child_mb_score,
                "full_score": full_total, "objectives": full_obj,
                "reflection_summary": reflection[:200],
            })
        else:
            print(f"    ✗ Rejected: child {child_mb_score:.0%} <= parent {parent_mb_score:.0%}")
            evolution_trace.append({
                "gen": gen, "task": task_type, "action": "rejected",
                "parent": parent_cid, "child": child_cid,
                "parent_mb_score": parent_mb_score, "child_mb_score": child_mb_score,
            })

    # ── (9) ATTEMPT MERGE after successful mutations ──
    for task_type in last_accepted:
        if random.random() > MERGE_PROBABILITY:
            continue

        frontier = frontiers[task_type]
        pair = frontier.find_merge_pair()
        if pair[0] is None:
            continue

        cid_a, cid_b = pair
        prompt_a = frontier.candidates[cid_a]["prompt"]
        prompt_b = frontier.candidates[cid_b]["prompt"]
        score_a = frontier.candidates[cid_a]["total"]
        score_b = frontier.candidates[cid_b]["total"]

        print(f"\n    [MERGE {task_type}] Combining {cid_a} ({score_a:.0%}) + {cid_b} ({score_b:.0%})")
        merged_prompt = merge_prompts(prompt_a, prompt_b, task_type)

        # Evaluate merged on full query set
        queries = EVAL_QUERIES[task_type]
        merged_results, merged_total, merged_obj = evaluate_candidate(merged_prompt, queries)
        parent_max = max(score_a, score_b)

        if merged_total >= parent_max:
            merged_cid = frontier.add_candidate(merged_prompt, merged_results, merged_total, merged_obj)
            print(f"    ✓ Merge accepted: {merged_total:.0%} >= max({score_a:.0%}, {score_b:.0%})")
            evolution_trace.append({
                "gen": gen, "task": task_type, "action": "merge_accepted",
                "parents": [cid_a, cid_b], "child": merged_cid,
                "merged_score": merged_total, "parent_max": parent_max,
            })
        else:
            print(f"    ✗ Merge rejected: {merged_total:.0%} < max({score_a:.0%}, {score_b:.0%})")
            evolution_trace.append({
                "gen": gen, "task": task_type, "action": "merge_rejected",
                "parents": [cid_a, cid_b], "merged_score": merged_total, "parent_max": parent_max,
            })

    # ── Generation summary ──
    print(f"\n  --- Gen {gen} Summary ---")
    for task_type in EVAL_QUERIES:
        frontiers[task_type].summary()
    print(f"  {eval_cache.stats()}")

## Section 8: Evolution Summary and Pareto Selection

In [ ]:
print("=" * 60)
print("EVOLUTION SUMMARY")
print("=" * 60)

final_prompts = {}
final_scores = {}

for task_type in EVAL_QUERIES:
    frontier = frontiers[task_type]
    best_cid, best_prompt = frontier.get_best_overall()
    best_data = frontier.candidates[best_cid]
    final_prompts[task_type] = best_prompt
    final_scores[task_type] = best_data["total"]

    print(f"\n{task_type}:")
    print(f"  Baseline: {baseline_scores[task_type]:.0%}")
    print(f"  Final:    {best_data['total']:.0%} (candidate {best_cid})")
    print(f"  Δ:        {best_data['total'] - baseline_scores[task_type]:+.0%}")
    print(f"  Objectives: fmt={best_data['objectives'].get('format', 0):.0%} "
          f"cnt={best_data['objectives'].get('content', 0):.0%} "
          f"cmp={best_data['objectives'].get('completeness', 0):.0%}")
    print(f"  Population: {len(frontier.candidates)} candidates explored")

    # Show lineage
    if best_cid in frontier.lineage:
        chain = [best_cid]
        current = best_cid
        while current in frontier.lineage:
            current = frontier.lineage[current]
            chain.append(current)
        print(f"  Lineage: {' → '.join(reversed(chain))}")

final_overall = sum(final_scores.values()) / len(final_scores)
print(f"\n{'=' * 60}")
print(f"Overall: {baseline_overall:.0%} → {final_overall:.0%} (Δ {final_overall - baseline_overall:+.0%})")
print(f"{eval_cache.stats()}")
print(f"{'=' * 60}")

## Section 9: View Final Optimized Prompts

In [ ]:
for task_type, prompt in final_prompts.items():
    print(f"\n{'=' * 60}")
    print(f"Task: {task_type} (Score: {final_scores[task_type]:.0%})")
    print(f"{'=' * 60}")
    print(prompt)

## Section 10: Save Results

In [ ]:
# Prepare serializable frontier data
frontier_export = {}
for task_type, frontier in frontiers.items():
    frontier_export[task_type] = {
        "candidates": {
            cid: {
                "prompt": data["prompt"],
                "total": data["total"],
                "objectives": data["objectives"],
                "per_query": data["per_query"],
            }
            for cid, data in frontier.candidates.items()
        },
        "instance_front": {
            qid: {"score": score, "candidate": cid}
            for qid, (score, cid, _) in frontier.instance_front.items()
        },
        "objective_front": {
            obj: {"score": score, "candidate": cid}
            for obj, (score, cid, _) in frontier.objective_front.items()
        },
        "lineage": frontier.lineage,
    }

output = {
    "metadata": {
        "method": "GEPA-faithful autonomous prompt optimization",
        "model": MODEL_ID,
        "generations": NUM_GENERATIONS,
        "population_size": POPULATION_SIZE,
        "minibatch_size": MINIBATCH_SIZE,
        "merge_probability": MERGE_PROBABILITY,
        "epsilon_greedy": EPSILON_GREEDY,
        "timestamp": datetime.now().isoformat(),
        "baseline_score": baseline_overall,
        "final_score": final_overall,
        "cache_stats": eval_cache.stats(),
    },
    "optimized_prompts": final_prompts,
    "scores": final_scores,
    "seed_prompts": SEED_PROMPTS,
    "evolution_trace": evolution_trace,
    "pareto_frontiers": frontier_export,
}

with open(OUTPUT_FILE, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved to {OUTPUT_FILE}")

## Section 11: Download Results

In [ ]:
# Uncomment to download from Colab
# from google.colab import files
# files.download(OUTPUT_FILE)